Clinical QA Grounding System

Preporcessing

In [1]:
# Import necessary libraries and mount your drive, upload necessary files according to the requirement
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [2]:
# Import necessary libraries and load admissions dataset
import pandas as pd
admissions = pd.read_csv('/content/drive/My Drive/ADMISSIONS.csv')

In [3]:
# Load diagnosis dataset
diagnosis = pd.read_csv('/content/drive//My Drive/Diagnosis.csv', sep='\t', encoding='UTF-16', usecols=["ROW_ID", "SUBJECT_ID", "HADM_ID", "SEQ_NUM", "ICD9CODE"])

In [4]:
# Load Procedure's dataset
procedures = pd.read_csv("/content/drive/My Drive/Procedures.csv", sep="\t", encoding="UTF-16",low_memory=False)

In [5]:
# Load Notevents Dataset
events = pd.read_csv("/content/drive/My Drive/NOTEEVENTS.csv",low_memory=False)

In [6]:
#Load Patients and prescriptions dataset
patients = pd.read_csv("/content/drive/My Drive/PATIENTS.csv")
prescriptions = pd.read_csv("/content/drive/My Drive/PRESCRIPTIONS.csv",low_memory=False)

In [7]:
# Import necessary libraries
import pandas as pd
import re
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
import spacy

# Drop rows with missing values in crucial fields
events = events.dropna(subset=['TEXT'])
diagnosis = diagnosis.dropna(subset=['ICD9CODE'])
procedures = procedures.dropna(subset=['ICD9CODE'])
prescriptions = prescriptions.dropna(subset=['DRUG'])
admissions = admissions.dropna(subset=['HADM_ID'])

In [8]:
# Drop the columns which are not required for the task to avoid confusion
events = events.drop(columns=['STORETIME', 'CGID', 'CHARTTIME'])

In [9]:
# Drop the null values
events = events.dropna(subset=['HADM_ID'])

In [11]:
events["ISERROR"] = events["ISERROR"].fillna(0)

In [10]:
# Extract Admission Date & Discharge Date correctly
events["ADMISSION_DATE"] = events["TEXT"].str.extract(r'Admission Date:\s*\[\*\*(\d{4}-\d{1,2}-\d{1,2})\*\*\]')
events["DISCHARGE_DATE"] = events["TEXT"].str.extract(r'Discharge Date:\s*\[\*\*(\d{4}-\d{1,2}-\d{1,2})\*\*\]')

In [12]:
# Change the column name to merge them
procedures.rename(columns={
    "SUBJECTID": "SUBJECT_ID",
    "HADMID": "HADM_ID"
}, inplace=True)

In [13]:
# Ensure matching dtypes for merge keys
events["SUBJECT_ID"] = events["SUBJECT_ID"].astype(str)
events["HADM_ID"] = events["HADM_ID"].astype(str)
admissions["SUBJECT_ID"] = admissions["SUBJECT_ID"].astype(str)
admissions["HADM_ID"] = admissions["HADM_ID"].astype(str)
patients["SUBJECT_ID"] = patients["SUBJECT_ID"].astype(str)
diagnosis["SUBJECT_ID"] = diagnosis["SUBJECT_ID"].astype(str)
diagnosis["HADM_ID"] = diagnosis["HADM_ID"].astype(str)
procedures["SUBJECT_ID"] = procedures["SUBJECT_ID"].astype(str)
procedures["HADM_ID"] = procedures["HADM_ID"].astype(str)
prescriptions["SUBJECT_ID"] = prescriptions["SUBJECT_ID"].astype(str)
prescriptions["HADM_ID"] = prescriptions["HADM_ID"].astype(str)

In [14]:
#install everytime when your runtine disconnects.
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 26.9 MB/s eta 0:00:00


In [15]:
#install everytime when your runtine disconnects.
!pip install openai --upgrade
!pip install nltk pandas numpy faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 603.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 31.1 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitl

In [ ]:
# set the environments
import os
os.environ["OPENAI_API_KEY"] = "Your key here"

In [17]:
#install everytime when your runtine disconnects.
!pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 4.3 MB/s eta 0:00:00


Implementation of core functionality

In [4]:
import os
import openai
from openai import OpenAI
from transformers import T5Tokenizer, T5ForConditionalGeneration
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize

# Setup
nltk.download('punkt_tab')
openai_client = OpenAI()

# Load models
retrieval_model = SentenceTransformer('all-MiniLM-L6-v2')
tokenizer_t5 = T5Tokenizer.from_pretrained('t5-small')
model_t5 = T5ForConditionalGeneration.from_pretrained('t5-small')

# Load and process clinical notes
def load_clinical_notes(file_path, max_notes=10000):
    data = pd.read_csv(file_path, low_memory=False)
    clinical_notes = data['TEXT'].dropna().tolist()
    return clinical_notes[:max_notes]

def encode_sentences(sentences, batch_size=32):
    return retrieval_model.encode(sentences, convert_to_numpy=True, batch_size=batch_size, show_progress_bar=True)

def build_faiss_index(embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings.astype(np.float32))
    return index

def retrieve_relevant_sentences(query, all_sentences, faiss_index, top_k=8):
    q_embed = retrieval_model.encode([query], convert_to_numpy=True)
    _, indices = faiss_index.search(q_embed.astype(np.float32), top_k)
    return [all_sentences[i] for i in indices[0]]

# Generate answer using OpenAI API with citations
def generate_openai_answer(clinician_q, patient_q, top_sentences):
    top_sentences = [s.strip() for s in top_sentences if len(s.strip()) > 10]
    context_block = "\n".join([f"{i+1}: {sent}" for i, sent in enumerate(top_sentences)])

    user_prompt = (
        f"You are a clinical assistant tasked with answering a medical question. "
        f"Given the sentences from clinical notes, generate a professional answer that clearly explains the reason for the treatment or event. "
        f"Cite evidence from the notes using numbered sentence references like (1), (2). Limit your answer to 75 words.\n\n"
        f"Context:\n{context_block}\n\n"
        f"Patient Question: {patient_q}\n"
        f"Clinician Question: {clinician_q}\n\n"
        f"Answer:"
    )

    response = openai_client.chat.completions.create(
        model="gpt-4",
        messages=[
            {"role": "system", "content": "You are a medical assistant who generates concise, professional explanations with citations from clinical notes."},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.3,
        max_tokens=200
    )

    answer = response.choices[0].message.content.strip()

    # Truncate to 75 words
    words = answer.split()
    if len(words) > 75:
        answer = " ".join(words[:75]) + "..."

    return answer

# Main function to retrieve the notes from the clinical note dataset and generate a stuctured answer
def interactive_qa_with_openai_rag(file_path, max_notes=1000, max_sentences=5000):
    clinical_notes = load_clinical_notes(file_path, max_notes)

    print("Splitting sentences...")
    all_sentences = []
    for note in clinical_notes:
        sentences = sent_tokenize(note.strip())
        all_sentences.extend(sentences)
        if len(all_sentences) >= max_sentences:
            break
    all_sentences = all_sentences[:max_sentences]
    print(f"Total sentences: {len(all_sentences)}")

    embeddings = encode_sentences(all_sentences)
    index = build_faiss_index(embeddings)

    patient_q = input("Enter the patient question:\n").strip()
    clinician_q = input("Enter the clinician question:\n").strip()

    combined_query = patient_q + " " + clinician_q
    top_sentences = retrieve_relevant_sentences(combined_query, all_sentences, index, top_k=8)

    print("\nTop Retrieved Sentences:")
    for i, sent in enumerate(top_sentences, 1):
        print(f"({i}): {sent}")

    answer = generate_openai_answer(clinician_q, patient_q, top_sentences)

    print("\nGenerated Answer with Citations (Max 75 Words):")
    print(answer)
    return answer, top_sentences, clinician_q, patient_q

# Run the RAG QA pipeline
file_path = '/content/drive/MyDrive/NOTEEVENTS.csv'
answer, top_sentences, clinician_q, patient_q = interactive_qa_with_openai_rag(file_path, max_notes=10000, max_sentences=20000)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read th

Splitting sentences...
Total sentences: 20000


Batches:   0%|          | 0/625 [00:00<?, ?it/s]

Enter the patient question:
I had severe abdomen pain and was hospitalised for 15 days in ICU, diagnoised with CBD sludge. Doctor advised for ERCP. My question is if the sludge was there does not any medication help in flushing it out? Whether ERCP was the only cure?
Enter the clinician question:
Why was ERCP recommended over a medication-based treatment for CBD sludge?

Top Retrieved Sentences:
(1): Patient underwent ERCP
in MICU while intubated, which demonstrated dilated CBD but no
stones.
(2): ERCP was done on [**2198-4-6**],
with sphinceterotomy and CBD stent placed.
(3): An ERCP was performed
which showed gallstones obstructing your bile ducts.
(4): Unchanged 1-cm gallbladder wall polyp versus tumefactive
sludge.
(5): The patient was not acutely ill last night, so she was
admitted to the Medicine team, with plan for ERCP today.
(6): Post-ERCP, she
was admitted to the ICU with a diagnosis of cholangitis.
(7): 4) Probable sludge within the gallbladder.
(8): ERCP [**2157-2-1**]:
Fin

Implementation of UI

In [8]:
import gradio as gr

# === New preprocessing before Gradio ===
# Prepare all_sentences and index before defining the Gradio app
print("Preparing sentences and FAISS index for Gradio app...")

# Load clinical notes
clinical_notes = load_clinical_notes(file_path, max_notes=10000)

# Split into sentences
all_sentences = []
for note in clinical_notes:
    sentences = sent_tokenize(note.strip())
    all_sentences.extend(sentences)
all_sentences = all_sentences[:20000]  # Limit to 20,000 sentences

# Encode and build FAISS index
embeddings = encode_sentences(all_sentences)
index = build_faiss_index(embeddings)

print(f"Prepared {len(all_sentences)} sentences and FAISS index.")

# === Gradio-compatible function ===
def interactive_qa_with_openai_rag_ui(patient_q, clinician_q):
    combined_query = patient_q + " " + clinician_q
    top_sentences = retrieve_relevant_sentences(combined_query, all_sentences, index, top_k=8)
    answer = generate_openai_answer(clinician_q, patient_q, top_sentences)

    retrieved_sentences_str = "\n".join([f"({i+1}) {s}" for i, s in enumerate(top_sentences)])
    return retrieved_sentences_str, answer

# === Build the Gradio app ===
demo = gr.Interface(
    fn=interactive_qa_with_openai_rag_ui,
    inputs=[
        gr.Textbox(label="Patient Question"),
        gr.Textbox(label="Clinician Question")
    ],
    outputs=[
        gr.Textbox(label="Top Retrieved Sentences"),
        gr.Textbox(label="Generated Answer with Citations")
    ],
    title="Clinical QA Assistant (RAG + GPT-4)",
    description="Enter a patient question and a clinician question. Retrieves relevant notes and generates a professional answer with citations."
)

# === Launch Gradio ===
demo.launch()

Preparing sentences and FAISS index for Gradio app...


Batches:   0%|          | 0/625 [00:00<?, ?it/s]

Prepared 20000 sentences and FAISS index.
It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://551d0bff77fc8a72de.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Installation of packages for evaluation

In [9]:
#install everytime when your runtine disconnects.
# BERTScore
!pip install bert-score

# ROUGE scorer
!pip install rouge-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=6449b821e7e2ad17762b465e6826c507a9349e7d8f8497091eb9fcca61aeec14
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge-score


Evaluation Script

In [11]:
# Import necessary libraries
from google.colab import drive
drive.mount('/content/drive')
import json

qa_id = 12  # Update this to your actual QA pair ID

# Load test.final.json from Google Drive
with open("/content/test.final.json") as f:
    test_data = json.load(f)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Evaluation is done using the test.final.json file which was provided to compare against the generated answers and the given cases.

In [12]:
# Import libraries
import torch
from bert_score import score as bert_score
from rouge_score import rouge_scorer
from sentence_transformers import util

# Load test.final.json
with open("/content/test.final.json") as f:
    test_data = json.load(f)

# Flatten test.final.json into list of QA pairs
def flatten_qa_pairs(test_data):
    all_qas = []
    for article in test_data["data"]:
        for para in article["paragraphs"]:
            for qa in para["qas"]:
                all_qas.append({
                    "id": qa["id"],
                    "question": qa["question"],
                    "answer": qa["answers"][0]["text"]
                })
    return all_qas

all_gold_qas = flatten_qa_pairs(test_data)

# Find best matching QA
def find_most_similar_qa(all_gold_qas, user_question, top_k=1):
    questions = [qa["question"] for qa in all_gold_qas]
    embeddings_gold = retrieval_model.encode(questions, convert_to_tensor=True)
    embedding_user = retrieval_model.encode(user_question, convert_to_tensor=True)
    similarities = util.pytorch_cos_sim(embedding_user, embeddings_gold)[0]
    top_result = torch.topk(similarities, k=top_k)
    idx = top_result.indices[0].item()
    return all_gold_qas[idx]

# Evaluate Factuality
def evaluate_factuality(generated_answer, gold_answer):
    P, R, F1 = bert_score([generated_answer], [gold_answer], lang="en", verbose=False)
    return {
        "BERTScore Precision": P.item(),
        "BERTScore Recall": R.item(),
        "BERTScore F1": F1.item()
    }

# Evaluate Relevance
def evaluate_relevance(generated_answer, top_sentences, labels=None):
    if not labels:
        labels = [1] * len(top_sentences)
    essential = [s for i, s in enumerate(top_sentences) if labels[i] == 2]
    if not essential:
        essential = [s for i, s in enumerate(top_sentences) if labels[i] == 1]
    reference = " ".join(essential)
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
    rouge = scorer.score(reference, generated_answer)
    P, R, F1 = bert_score([generated_answer], [reference], lang="en", verbose=False)
    return {
        "ROUGE-1 F1": rouge['rouge1'].fmeasure,
        "ROUGE-L F1": rouge['rougeL'].fmeasure,
        "BERTScore F1": F1.item()
    }

# Combine both patient and clinician questions for matching
combined_user_q = patient_q + " " + clinician_q
matched_gold = find_most_similar_qa(all_gold_qas, combined_user_q)
gold_answer = matched_gold["answer"]

print("\nMatched QA ID:", matched_gold["id"])

# Run evaluations
factuality_scores = evaluate_factuality(answer, gold_answer)
relevance_scores = evaluate_relevance(answer, top_sentences)

print("\n=== Factuality Evaluation ===")
for k, v in factuality_scores.items():
    print(f"{k}: {v:.4f}")

print("\n=== Relevance Evaluation ===")
for k, v in relevance_scores.items():
    print(f"{k}: {v:.4f}")



Matched QA ID: 568


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



=== Factuality Evaluation ===
BERTScore Precision: 0.8137
BERTScore Recall: 0.8249
BERTScore F1: 0.8193

=== Relevance Evaluation ===
ROUGE-1 F1: 0.3141
ROUGE-L F1: 0.1675
BERTScore F1: 0.8369


**References** : <br>
[1] BioNLP shared task., ArchEHR-QA 2025 (pronounced "Archer"):Shared Task on Grounded Electronic Health Record Question Answering <br>
[2] A Neural Approach to Answering Medical Questions
 Gupta, P., & Singh, M. (2021). A neural approach to answering medical questions. ACM Digital Library. Retrieved from https://dl.acm.org/doi/10.1145/3490238. <br>
[3] Question Answering for Electronic Health Records: Scoping Review of Datasets and Models
 Miotto, R., Li, L., Kidd, B. A., & Dudik, M. (2021). Question answering for electronic health records: Scoping review of datasets and models. National Library of Medicine. Retrieved from https://pmc.ncbi.nlm.nih.gov/articles/PMC11561445/